In [0]:
dbutils.widgets.text("catalog","fk_1392")
catalog = dbutils.widgets.get("catalog")

#orders

In [0]:
orders_df = spark.table(f"{catalog}.bronze.orders")

In [0]:
from pyspark.sql import functions as F

#standarization
orders_df = orders_df.withColumn("order_id",F.upper("order_id")) \
    .withColumn("customer_id",F.upper("customer_id")) \
    .withColumn("store_id",F.upper("store_id")) \
    .withColumn("order_ts",F.to_timestamp("order_ts")) \
    .withColumn("updated_at",F.to_timestamp("updated_at")) \
    .withColumn("order_status",F.upper(F.trim("order_status"))) \
    .withColumn("payment_method",F.upper(F.col("payment").getItem("method"))) \
    .withColumn("orderdate",F.date_format("order_ts","yyyy-MM-dd")) 


#validation
orders_df = orders_df.withColumn("order_status",F.when(F.col("order_status").isin("CANCELLED","CANCELED"),"CANCELLED")
                                .when(F.col("order_status").isin("DELEVERED","DELEVER"),"DELEVERED")
                                .otherwise(F.col("order_status")))



#orphan customer 
valid_customer_id = [row[0] for row in spark.table(f"{catalog}.silver.customers").select("customer_id").distinct().collect()]
#orphan store
valid_store_id = [row[0] for row in spark.table(f"{catalog}.silver.stores").select("store_id").distinct().collect()]

valid_procduct_id = [row[0] for row in spark.table(f"{catalog}.silver.products").select("product_id").distinct().collect()]
#invalid timestamp
#empty order


orders_df  = orders_df.withColumn("validation_error",F.concat_ws(",",
                    F.when(F.col("customer_id").isNull(),"MISSING_CUSTOMER"),
                    F.when(~F.col("customer_id").isin(valid_customer_id),"INVALID_CUSTOMER"),
                    F.when(F.col("store_id").isNull() | ~F.col("store_id").isin(valid_store_id),"UNKNOWN_STORE"),
                    F.when(F.col("order_ts").isNull(),"MISSING_TIMESTAMP"),
                    F.when(F.col("order_ts") < F.to_timestamp(F.lit("2022-01-01 00:00:00")),"INVALID_TIMESTAMP"),
                    F.when(F.array_min(F.col("items.unit_price")) <= 0, "INVALID_PRICE"),
                    F.when(~F.col("order_status").isin("CANCELLED","DELEVERED","DELIVERED","PLACED"),"INVALID_STATUS"),
                    F.when(F.array_min(F.col("items.qty")) <= 0,"INVALID_QUANTITY"),
                    F.when(F.col("items").isNull() | (F.size(F.col("items"))==0),"EMPTY_ORDERS")           
                                                                
           ))
orders_df = orders_df.drop("ingestion_timestamp","source_file")

valid_orders_df = orders_df.filter(F.col("validation_error") == "")
invalid_orders_df = orders_df.filter(F.col("validation_error") != "")

invalid_orders_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_orders")

valid_orders_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders")


#store

In [0]:
store_df = spark.table(f"{catalog}.bronze.stores")

In [0]:
from pyspark.sql import functions as F

store_df = store_df.withColumn("store_id",F.upper("store_id")) \
    .withColumn("store_name",F.upper(F.trim("store_name"))) \
    .withColumn("city",F.initcap(F.trim("city"))) \
    .withColumn("state",F.trim("state")) \
    .withColumn("region",F.trim("region")) \
        .withColumn("opening_date",F.to_date("opening_date","yyyy-MM-dd"))

store_df =store_df.drop("ingestion_timestamp","source_file")

store_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stores")

#customers

In [0]:
customer_df = spark.table(f"{catalog}.bronze.customers")



In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.customers(
    customer_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    tier STRING,
    signup_date DATE,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN,
    version INT
)
""")

In [0]:
from pyspark.sql import functions as F
#standarization
customer_df = customer_df.withColumn("customer_id",F.upper("customer_id")) \
    .withColumn("full_name",F.trim(F.col("full_name"))) \
    .withColumn("email",F.lower(F.trim("email"))) \
    .withColumn("phone",F.regexp_replace(F.col("phone"),r"[^0-9]","")) \
    .withColumn("city",F.initcap(F.trim("city"))) \
    .withColumn("tier",F.upper(F.trim("tier"))) \
    .withColumn("signup_date",F.coalesce(F.try_to_date("signup_date","yyyy-MM-dd"),F.try_to_date("signup_date","dd/MM/yyyy"))) \
    .withColumn("effective_from",F.current_date()) 


customer_df.createOrReplaceTempView("customer_view")


In [0]:
spark.sql(f"""
MERGE INTO {catalog}.silver.customers t
USING customer_view s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.is_current = true 
THEN UPDATE SET 
is_current = FALSE,
effective_to = s.effective_from
""")



In [0]:
spark.sql(f"""        
    INSERT INTO {catalog}.silver.customers
    SELECT
        s.customer_id,
        s.full_name,
        s.email,
        s.phone,
        s.city,
        s.tier,
        s.signup_date,
        s.effective_from,
        NULL,
        TRUE,
        COALESCE((SELECT MAX(t.version) FROM {catalog}.silver.customers t WHERE t.customer_id = s.customer_id), 0) + 1
    FROM customer_view s
    WHERE NOT EXISTS (
        SELECT 1 FROM {catalog}.silver.customers t WHERE t.customer_id = s.customer_id AND t.is_current = true
    )  
""")

#products

In [0]:
product_df = spark.table(f"{catalog}.bronze.products")

In [0]:
from pyspark.sql import functions as F

product_df = product_df.withColumn("product_id",F.upper(F.trim("product_id"))) \
    .withColumn("category_l1",F.col("category.l1"))\
    .withColumn("category_l2",F.col("category.l2"))\
.withColumn("pack_size",F.col("pack.size"))\
.withColumn("pack_unit",F.col("pack.unit")) 
    
product_df = product_df.drop("ingestion_timestamp","source_file")



In [0]:
product_df.createOrReplaceTempView("product_view")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.products(
    product_id STRING,
    product_name STRING,
    brand string,
    category_l1 STRING,
    category_l2 STRING,
    pack_size INT,
    pack_unit STRING,
    mrp DECIMAL(10,2),
    selling_price DECIMAL(10,2),
    is_active Boolean,
    tags ARRAY<STRING>,
    last_changed_snapshot_date DATE
)
""")

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.silver.products t
USING product_view s
ON t.product_id = s.product_id
WHEN MATCHED AND
t.product_name != s.product_name OR
t.brand != s.brand OR
t.category_l1 != s.category_l1 OR
t.category_l2 != s.category_l2 OR
t.pack_size != s.pack_size OR
t.pack_unit != s.pack_unit OR
t.mrp != s.mrp OR
t.selling_price != s.selling_price OR
t.is_active != s.is_active OR
t.tags != s.tags 
THEN UPDATE SET 
    t.product_name = s.product_name,
    t.brand = s.brand,
    t.category_l1 = s.category_l1,
    t.category_l2 = s.category_l2,
    t.pack_size = s.pack_size,
    t.pack_unit = s.pack_unit,
    t.mrp = s.mrp,
    t.selling_price = s.selling_price,
    t.is_active = s.is_active,
    t.tags = s.tags,
    t.last_changed_snapshot_date = s.snapshot_date
WHEN NOT MATCHED 
THEN INSERT 
(product_id,product_name,brand,category_l1,category_l2,pack_size,pack_unit,mrp,selling_price,is_active,tags,last_changed_snapshot_date) VALUES
(s.product_id,s.product_name,s.brand,s.category_l1,s.category_l2,s.pack_size,s.pack_unit,s.mrp,s.selling_price,s.is_active,s.tags,s.snapshot_date)               
""")

In [0]:
spark.sql(f"""
ALTER TABLE {catalog}.silver.products
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")